# Kinship Library
## Notebook 03: K Means, PCA and the First Reading Path Recommender

### Goal

Explore the structure of the catalogue and build the first combined recommendation prototype.

### Questions

1. Do TF IDF features form interpretable clusters?
2. What number of clusters is reasonably supported by inertia and silhouette score?
3. What does PCA reveal about the high dimensional catalogue?
4. How different are TF IDF and semantic recommendations?
5. Can we return a small reading path instead of pretending there is one perfect book?

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"
MODELS = PROJECT_ROOT / "models"

books = pd.read_csv(
    PROCESSED_DATA / "books_nlp_ready.csv"
)

book_embeddings = np.load(
    MODELS / "book_embeddings.npy"
)

print("Books:", len(books))
print("Embeddings:", book_embeddings.shape)

## 1. Rebuild the TF IDF representation

We rebuild it from the saved clean text so this notebook can run independently after Notebook 02.

In [ ]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2
)

tfidf_matrix = tfidf.fit_transform(
    books["clean_text"].fillna("")
)

print(tfidf_matrix.shape)

## 2. Test K values with elbow and silhouette score

In [ ]:
k_values = range(2, 11)
inertias = []
silhouette_scores = []

for k in k_values:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(tfidf_matrix)

    inertias.append(model.inertia_)

    silhouette_scores.append(
        silhouette_score(
            tfidf_matrix,
            labels
        )
    )

plt.figure(figsize=(8, 5))
plt.plot(
    list(k_values),
    inertias,
    marker="o"
)
plt.title("Elbow method")
plt.xlabel("Number of clusters")
plt.ylabel("Inertia")
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(
    list(k_values),
    silhouette_scores,
    marker="o"
)
plt.title("Silhouette score")
plt.xlabel("Number of clusters")
plt.ylabel("Silhouette score")
plt.show()

## 3. Select K

The highest silhouette score gives us an objective starting point.

We still inspect whether the resulting clusters make sense.

In [ ]:
best_k = list(k_values)[
    int(np.argmax(silhouette_scores))
]

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

books["cluster"] = kmeans.fit_predict(
    tfidf_matrix
)

print("Selected K:", best_k)

books["cluster"].value_counts().sort_index()

## 4. Interpret the clusters

The top TF IDF terms around each centroid help us understand the vocabulary that defines each group.

We do not automatically rename a cluster as a cultural or spiritual tradition.

In [ ]:
terms = np.array(
    tfidf.get_feature_names_out()
)

order_centroids = (
    kmeans.cluster_centers_
    .argsort()[:, ::-1]
)

for cluster_number in range(best_k):

    top_terms = terms[
        order_centroids[
            cluster_number,
            :12
        ]
    ]

    print(
        "Cluster",
        cluster_number,
        ":",
        ", ".join(top_terms)
    )

## 5. PCA visualization

PCA reduces the TF IDF representation to two dimensions so we can visualize it.

The two components usually preserve only part of the original information. We report the explained variance rather than treating the plot as the complete structure of the data.

In [ ]:
pca = PCA(
    n_components=2,
    random_state=42
)

coordinates = pca.fit_transform(
    tfidf_matrix.toarray()
)

books["pca_1"] = coordinates[:, 0]
books["pca_2"] = coordinates[:, 1]

plt.figure(figsize=(9, 6))

scatter = plt.scatter(
    books["pca_1"],
    books["pca_2"],
    c=books["cluster"],
    alpha=0.7
)

plt.title("PCA view of Kinship Library")
plt.xlabel("PCA component 1")
plt.ylabel("PCA component 2")
plt.colorbar(scatter, label="Cluster")
plt.show()

print(
    "Explained variance:",
    round(
        pca.explained_variance_ratio_.sum(),
        3
    )
)

## 6. Semantic recommender

The semantic embeddings were produced in Notebook 02.

We load the same Sentence Transformer model so that new reader queries live in the same vector space as the saved book embeddings.

In [ ]:
# Uncomment in Colab if needed
# !pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

semantic_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
def recommend_semantic(user_text, number_of_books=10):

    query_embedding = semantic_model.encode(
        [user_text],
        normalize_embeddings=True
    )

    scores = cosine_similarity(
        query_embedding,
        book_embeddings
    ).flatten()

    best_indices = scores.argsort()[::-1][:number_of_books]

    results = books.iloc[best_indices][
        [
            "title",
            "author",
            "collection_theme",
            "cluster"
        ]
    ].copy()

    results["semantic_score"] = scores[best_indices]

    return results.reset_index(drop=True)

## 7. Reading path prototype

Instead of returning only the highest scoring books, we can select from the top semantic candidates while encouraging some thematic variety.

This is a transparent rule based prototype, not a claim that one tradition is the correct answer to a reader's concern.

In [ ]:
def create_reading_path(user_text, number_of_books=5):

    candidates = recommend_semantic(
        user_text,
        number_of_books=20
    )

    selected_rows = []
    used_themes = set()

    for _, row in candidates.iterrows():

        theme = row["collection_theme"]

        if theme not in used_themes:
            selected_rows.append(row)
            used_themes.add(theme)

        if len(selected_rows) == number_of_books:
            break

    if len(selected_rows) < number_of_books:

        selected_titles = {
            row["title"]
            for row in selected_rows
        }

        for _, row in candidates.iterrows():

            if row["title"] not in selected_titles:
                selected_rows.append(row)
                selected_titles.add(row["title"])

            if len(selected_rows) == number_of_books:
                break

    return pd.DataFrame(selected_rows).reset_index(drop=True)

In [ ]:
example_reader_text = (
    "I feel disconnected from nature and worried about the future. "
    "I want to understand how to live with more care, meaning, "
    "spiritual depth, and connection with the living world."
)

create_reading_path(
    example_reader_text,
    number_of_books=5
)

## 8. First evaluation table

Recommendation systems need evaluation.

For a bootcamp project, we can create a small human reviewed benchmark.

For each reader query, inspect the top recommendations and mark relevance from zero to two.

Zero means not relevant.

One means partly relevant.

Two means strongly relevant.

This gives us a simple way to compare TF IDF and semantic retrieval without pretending that similarity score alone proves usefulness.

In [ ]:
evaluation_template = pd.DataFrame({
    "reader_query": [
        "disconnection from nature",
        "ecological concern",
        "search for meaning",
        "embodied way of living"
    ],
    "book_title": ["", "", "", ""],
    "model": ["semantic", "semantic", "semantic", "semantic"],
    "human_relevance_0_to_2": [np.nan, np.nan, np.nan, np.nan],
    "notes": ["", "", "", ""]
})

evaluation_template

## 9. Save modeled catalogue

SQL and Streamlit can now work from a stable modeled dataset rather than rebuilding the NLP pipeline every time.

In [ ]:
books.to_csv(
    PROCESSED_DATA / "books_modeled.csv",
    index=False
)

print(
    "Saved:",
    PROCESSED_DATA / "books_modeled.csv"
)